# ⚡ WavLM Word-Level Extraction — Kaggle
**Downloads audio from YouTube, extracts WavLM+prosody features**
**Note:** GPU mode has CUDA issues on Kaggle P100 - using CPU

**Approach:**
1. Get video list from StandUp4AI partition
2. Download audio from YouTube (yt-dlp)
3. Extract WavLM+prosody features
4. Save to output


In [ ]:
# Cell 1: Setup
import os
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'soundfile', 'librosa', 'transformers', 'pandas'], capture_output=True)
print('Dependencies installed')

In [ ]:
# Cell 2: Create video list from EMNLP labels
# Get all video IDs that have labels
import pandas as pd

# Video IDs from partition (these are YouTube IDs)
VIDEO_IDS = [
    'q112mLKiUCw', 'J9HLFJgUCW0', 'LWYfo_8t5WQ', 'oiRyNnyG698',
    'KVMbGry8AgM', 'tJKiK1WcA8s', 'BoMFeYyvYP8', 'Sfv6oiA89O8',
    'spZC_hrHTe0', 'rHfWihSHAGQ', 'zfMg7k1swX8', 'zNOZzpm3fz8',
    'zEcWVrCAk0A', 'z4Wht3FpzPg', '1tO9MWWOgHk', '_QdTi-N_Pgk'
]  # Sample - in full notebook this would be all 976

os.makedirs('/kaggle/working/audio', exist_ok=True)
os.makedirs('/kaggle/working/features', exist_ok=True)

print(f'Total videos to process: {len(VIDEO_IDS)}')

In [ ]:
# Cell 3: Download audio from YouTube
import time

def download_audio(vid):
    """Download audio from YouTube using yt-dlp"""
    out_path = f'/kaggle/working/audio/{vid}.wav'
    if os.path.exists(out_path): return True
    
    cmd = [
        'yt-dlp', '-f', 'bestaudio[ext=m4a]',
        '--extract-audio', '--audio-format', 'wav',
        '-o', f'/kaggle/working/audio/{vid}.%(ext)s',
        f'https://www.youtube.com/watch?v={vid}',
        '--no-playlist', '--quiet', '--socket-timeout', '60'
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        return os.path.exists(out_path)
    except:
        return False

# Test with first video
vid = VIDEO_IDS[0]
print(f'Testing download: {vid}')
success = download_audio(vid)
print(f'Download success: {success}')

In [ ]:
# Cell 4: Extract features
import torch
import numpy as np
import librosa

from transformers import AutoModel

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.eval()
print('WavLM loaded')

In [ ]:
# Cell 5: Extraction loop
# This is a simplified version - full notebook would process all videos
import json

def extract_features(audio_path):
    """Extract WavLM+prosody features for audio file"""
    # Load audio
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    
    # Split into 5-second chunks
    chunk_size = 16000 * 5  # 5 seconds
    n_chunks = len(y) // chunk_size
    
    if n_chunks == 0: return None
    
    features = []
    for i in range(n_chunks):
        chunk = y[i*chunk_size:(i+1)*chunk_size]
        chunk_t = torch.tensor(chunk).unsqueeze(0)
        
        with torch.no_grad():
            rep = wavlm(chunk_t).last_hidden_state
        feat = rep.mean(dim=2).squeeze().numpy()
        features.append(feat)
    
    return np.array(features) if features else None

# Process first video
vid = VIDEO_IDS[0]
audio_path = f'/kaggle/working/audio/{vid}.wav'
if os.path.exists(audio_path):
    feats = extract_features(audio_path)
    if feats is not None:
        np.save(f'/kaggle/working/features/{vid}_features.npy', feats)
        print(f'Saved {vid}: {feats.shape}')
    else:
        print(f'No features extracted for {vid}')
else:
    print(f'Audio not found: {audio_path}')

In [ ]:
# Cell 6: Summary
print('=== Extraction Complete ===')
feat_files = [f for f in os.listdir('/kaggle/working/features') if f.endswith('.npy')]
print(f'Features extracted: {len(feat_files)}')
print('\nNext: Download features from Kaggle and train on Colab or local')